In [ ]:
import re
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter, defaultdict

In [ ]:
# 载入文件
raw_data = pd.read_csv('data/train.csv')
raw_test_data = pd.read_csv('data/test.csv')
with open("data/chinese_word_freq_list.txt", encoding='utf-8') as f:
    words_freq = f.readlines()
raw_words_freq_dict = [(line.strip().split()[1], int(line.strip().split()[2])) for line in words_freq]
raw_data.head(), raw_words_freq_dict[:5]

In [ ]:
# 提取
sentences = raw_data.iloc[:, 0].values
words = [i.split('  ') for i in sentences]
sentences[:5], words[:5]

In [ ]:
# 统计
flat_words = [word for sentence in words for word in sentence]
counter = Counter(flat_words)
raw_words_dict = list(counter.items())
flat_words[:10], raw_words_dict[:10]

In [ ]:
# 构建训练集、测试集
train_data = [''.join(i.split('  ')) for i in sentences]
train_label=[sen.replace("  ", " ") for sen in sentences]
test_data = raw_test_data.iloc[:, 1].values
train_data[:5], train_label[:5]

In [ ]:
# 构建总词典, 滤去低频词
threshold = 0
# freqFile
words_freq_dict = [i for i in raw_words_freq_dict if i[1]>threshold]
total_count_freq = sum(count for _, count in words_freq_dict)
vocab_size_freq = len(words_freq_dict)
words_dict = [(i[0], (i[1] + 1) / (total_count_freq + vocab_size_freq)) for i in words_freq_dict]
print(len(words_freq))
# 学习所得
words_dict = [i for i in raw_words_dict if i[1]>threshold]
total_count = sum(count for _, count in words_dict)
vocab_size = len(words_dict)
words_dict = [(i[0], (i[1] + 1) / (total_count + vocab_size)) for i in words_dict]
print(len(words))
# 最终表
words_all_withfreq = words_dict
word_all_withfreq = defaultdict(int)
for word, freq in words_all_withfreq:
    word_all_withfreq[word] += freq
words_all_withfreq_dict = dict(words_all_withfreq)
words_all = [i[0] for i in list(words_all_withfreq_dict.items())]
# 最长的有多长(方便设置max_len)
lens = [len(i) for i in words_all]
print('lens', np.max(lens))
words_all_withfreq[:10], len(words_all)

In [ ]:
# 构建部分有固定模式未登录词的有限状态机
patterns = [
    # 全角英文单词
    r"[\uFF21-\uFF3A\uFF41-\uFF5A]+",
    # 年月日
    r"([０-９]+年[０-９]+月[０-９]+日)",
    # 年?月日?
    r"(([０-９]+年)?[０-９]+月([０-９]+日)?)",
    # 年-年
    r"([０-９]+-[０-９]+)",
    # 百分数
    r"([０-９]+(\.[０-９]+)?)%",
    # 带单位的数字
    r'([０-９]+(\.[０-９]+)?)\s*(米|厘米|千米|公里|公斤|千克|克|秒|分钟|小时|天|家|多家|万家|个|年|张|位)'
    # 数字串（电话号码等）
    r"([０-９]+)",
]

def FSM(sentence, words=[], direction="forward", patterns=patterns):
    '''
    sentence:要分的句子
    words:前面已经分好的词（按理说不应该在这里出现）
    direction:匹配句首还是句尾（正向还是反向）
    patterns:模式库
    返回words,sentence或未匹配到(False)
    '''
    for pattern in patterns:
        if direction=='forward':
            pattern = '^'+pattern
        else:
            pattern = pattern+'$'
        match = re.match(pattern, sentence)
        # print(pattern, sentence)
        if match:
            word = match.group()
            words.append(word)
            if direction=='forward':
                sentence = sentence[len(word):]
            else:
                sentence = sentence[:-len(word)]
            return words, sentence
        else:
            continue
    return False

# 看看构建成功了没
FSM([], "１９９７６.２０%")

In [ ]:
# 构建相邻前向词字典
prev_words = defaultdict(set)
for sentence in tqdm(words):
    for i in range(1, len(sentence)):
        curr_word = sentence[i]
        prev_word = sentence[i - 1]
        prev_words[curr_word].add(prev_word)
for word in prev_words:
    prev_words[word] = list(set(prev_words[word]))
prev_words

In [ ]:
# 正向最大匹配
def cut_words_forward(sentence, dict_words, max_len=5):
    dict_words = set(dict_words)
    words = []
    while len(sentence) > 0:
        unknown = FSM(sentence, words, "forward")
        # print(unknown)
        if unknown != False:
            words, sentence = unknown
            # print('append', words[-1])
            continue
        for i in range(max_len, 0, -1):
            if sentence[:i] in dict_words:
                # print(sentence[:i])
                words.append(sentence[:i])
                sentence = sentence[i:]
                break
            elif i == 1:
                words.append(sentence[:1])
                sentence = sentence[1:]
                break
    return words

# 单独尝试前向
trial=cut_words_forward("在８８３年，唐统治者纠集军队和地主武装向农民军反扑，农民军被迫撤出长安。", words_all, max_len=5)
print(" ".join(trial))

In [ ]:
# 逆向最大匹配
def cut_words_backward(sentence, dict_words, max_len=5):
    dict_words = set(dict_words)
    words = []
    length = len(sentence)-1
    while length >= 0:
        length = len(sentence)-1
        unknown = FSM(sentence, words, "backward")
        if unknown != False:
            words, sentence = unknown
            continue
        for i in range(max_len-1, 0, -1):
            start_idx = length-i if length-i>=0 else 0
            if sentence[start_idx:] in dict_words:
                # print(sentence[start_idx:])
                words.append(sentence[start_idx:])
                sentence = sentence[:start_idx]
                break
            elif i == 1:
                words.append(sentence[length:])
                sentence = sentence[:length]
    return words[::-1]

trial=cut_words_backward("是能够在未来社会中站住脚跟，开创事业的人才。", words_all, max_len=5)
print(" ".join(trial))

In [ ]:
# 正反向结合
def cut_words(sentence, dict_words, pdict, max_len=5, weight=10, prev_words=prev_words):
    '''
    已知：
    单个句子sentence
    存有所有词的列表dict_word（其实也没有必要有，可以直接从pdict获取）
    词-概率字典pdict（key=词，value=概率）
    双向结合前得：
    两个方向的索引-词映射变量fpos和bpos（即词位置字典，key=索引，value=词）
    一个空数组result
    步骤：
    1. 遍历fpos的key
    2. 如果fpos和bpos对应key处的value字符串长度相同（证明此处分的一样），则将该处字符串加入result数组；
        如果不同则进行以下操作：
        - 循环范围：分别遍历fpos和bpos，从该位置开始，到在fpos和bpos均能找到的第一个相同的key（如果没有则到末尾）
        - 循环内操作：将遍历过程中的词在pdict中对应的概率累乘（或取对数相加），并将该词对应放到fword或bword数组中暂存
        - 循环外比较：在两次遍历结束后对总概率fp和bp进行比较，如果fp大则将result数组与fword数组相拼，否则与bword相拼
    3. 下一次循环跳到在fpos和bpos均能找到的第一个相同的key处（新区域的第一个坐标肯定是相同的），如果没有则结束循环
    ''' 
    def get_word_positions(words):
        """将分词结果转换为索引 → 词映射"""
        start = 0
        positions = {}
        for word in words:
            positions[start] = word
            start += len(word)
        return positions
    fresult = cut_words_forward(sentence, dict_words, max_len)
    bresult = cut_words_backward(sentence, dict_words, max_len)
    fresult, bresult = fresult, bresult
    fpos = get_word_positions(fresult)
    bpos = get_word_positions(bresult)
    # print(fpos)
    # 初始化
    result = []
    fp, bp = 0,0
    fwords, bwords = [], []
    # 遍历fpos的key
    for i in fpos:
        # print(fpos[i])
        # 跳到第一个相同的key处
        if i not in bpos:
            continue
        # 分的一样
        if fpos[i] == bpos[i]:
            # print("append", fpos[i])
            result.append(fpos[i])
        # 分的不一样（最大原则+概率比较）
        else:
            # print("comp", fpos[i], bpos[i])
            start = i
            # 正向
            for fi in sorted(fpos):
                if fi < start:
                    continue
                if fi in bpos and fi != start:
                    break
                fword = fpos[fi]
                pword = pdict[fword] if fword in pdict else 1e-10
                # 前面一个词在前向词典中的话就激励
                if fwords!= []:
                    if fwords[-1] in prev_words[fword]:
                        pword=pword*weight
                elif result != [] and result[-1] in prev_words[fword]:
                    pword=pword*weight
                fp=fp+math.log(pword)
                fwords.append(fword)
            # 反向
            for bi in sorted(bpos):
                if bi < start:
                    continue
                if bi in fpos and bi != start:
                    break
                bword = bpos[bi]
                pword = pdict[bword] if bword in pdict else 1e-10
                # 前面一个词在前向词典中的话就激励
                if bwords!= []:
                    if bwords[-1] in prev_words[bword]:
                        pword=pword*weight
                elif result != [] and result[-1] in prev_words[bword]:
                    pword=pword*weight
                bp=bp+math.log(pword)
                bwords.append(bword)
            # print("append", fwords, bwords)
            # 有直接分成一个的就用一整块（词块最大化原则）
            if len(fwords) == 1:
                result.extend(fwords)
            elif len(bwords) == 1:
                result.extend(bwords)
            elif fp > bp:
                # print(fp, fwords)
                # print(bp, bwords)
                result.extend(fwords)
            else:
                result.extend(bwords)
            fp, bp = 1, 1
            fwords, bwords = [], []
    return result

trial=cut_words("在石山区、荒漠区", words_all, words_all_withfreq_dict, max_len=50)
print(" ".join(trial))

In [ ]:
# 随便构造点例子试试
# trial = cut_words_backward("另外旅游、侨汇也是", words_all, max_len=5)
idx = random.randint(0, len(train_data)-1)
trial = cut_words(train_data[idx], words_all, words_all_withfreq_dict, max_len=50)
print(" ".join(trial))

In [ ]:
# 在训练集前3000个上试试（部分交集歧义因最大匹配算法的贪婪性无法解决）
train_ans = []
for i in tqdm(train_data[:3000]):
    train_ans.append(' '.join(cut_words(i, words_all, words_all_withfreq_dict, max_len=20)))

In [ ]:
# 计算分词效果
wrong = []
def mean_f_score(pred_sentences, true_sentences):
    """
    计算 Mean F-Score 用于评估分词效果
    参数：
    - pred_sentences: 预测分词的句子列表 (list of list of str)
    - true_sentences: 真实分词的句子列表 (list of list of str)
    返回：
    - Mean F1-score
    """
    pred_sentences = [sentence.split() for sentence in pred_sentences]
    true_sentences = [sentence.split() for sentence in true_sentences]
    assert len(pred_sentences) == len(true_sentences), "预测和真实数据长度不匹配！"
    def f1_score(pred, true):
        pred_set, true_set = set(pred), set(true)
        if pred_set-(pred_set&true_set): # 看看不一样的都是什么样的句子
            wrong.append([pred, true])
            # print("pred:", pred)
            # print("true:", true)
        correct = len(pred_set & true_set)  # 交集，即预测正确的分词
        p = correct / len(pred_set) if pred_set else 0
        r = correct / len(true_set) if true_set else 0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        return f1
    f1_scores = [f1_score(pred, true) for pred, true in zip(pred_sentences, true_sentences)]
    return sum(f1_scores) / len(f1_scores) if f1_scores else 0
idx = random.randint(0, len(train_ans)-1)
print(idx)
print(train_ans[idx])
print(train_label[idx])
mean_f_score(train_ans, train_label[:3000])

In [ ]:
for i,j in wrong: # 看看不一样的都是什么样的句子
    print(" ".join(i))
    print(" ".join(j))
# 以 人 为 本、林业部 门将

In [ ]:
# jieba分词（现有的分词：jieba、hanlp等，精度hanlp>jieba，速度hanlp<jieba)
# import jieba
# jieba_results = [' '.join(list(jieba.cut(sent))) for sent in train_data]
# print(jieba_results[:5])
# print(train_label[:5])
# mean_f_score(jieba_results, train_label)
# print(list(jieba.cut("首先是没有人出名，这件事情理之中")))